### Retrospective Labelling: Soft labelling of missing training labels

From Part 1 (tuning on pr-auc)
* Best Performing Model is LightGBM (performed the best on the holdout dataset (best PR_AUC, ROC_AUC, Precision, F1)), performance on Recall is onl second to CatBoost
* Best Parameters for LightGBM:
`{'model__scale_pos_weight': 3.0, 'model__num_leaves': 31, 'model__n_estimators': 500, 'model__min_child_samples': 50, 'model__max_depth': 20, 'model__learning_rate': 0.1}`

From Part 2 (tuning on log_loss)
* Best performing model is LightGBM (lowest holdout log loss, highest ROC-AUC, PR-AUC)
* Best parameters:
`{'model__scale_pos_weight': 3.0, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__min_child_samples': 20, 'model__max_depth': 10, 'model__learning_rate': 0.03}`

* Selected the hyperparameters of LightGBM (log-loss, pr-auc tuned) => using an ensemble weight of 70% log-loss, 30% pr-auc

In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier


In [3]:
train_raw = pd.read_csv("train_raw_node_info.csv")

model_name = "LightGBM 70/30 ensemble"

# Parameters selected previously using PR-AUC tuning.
pr_auc_params = {
    "scale_pos_weight": 3.0,
    "num_leaves": 31,
    "n_estimators": 500,
    "min_child_samples": 50,
    "max_depth": 20,
    "learning_rate": 0.1,
}

# Parameters selected previously using log-loss tuning.
log_loss_params = {
    "scale_pos_weight": 3.0,
    "num_leaves": 15,
    "n_estimators": 500,
    "min_child_samples": 20,
    "max_depth": 10,
    "learning_rate": 0.03,
}

# Selected ensemble weights.
weight_log_loss = 0.70
weight_pr_auc = 0.30

#remove node_cols, overall info, when training for prediction, need a different way to use node_cols which do not break time leakage
node_cols = [
    "in_degree",
    "out_degree",
    "total_degree",
    "degree_ratio",
    "timestep_tx_volume",
]


In [4]:
def get_X_for_ids(df, ids):
    indexed = df.set_index("txId", drop=False)
    aligned = indexed.loc[list(ids)]
    excluded = {"txId", "index", "label"}
    feature_columns = [
        c for c in aligned.select_dtypes(include=np.number).columns
        if c not in excluded
    ]
    return aligned[feature_columns]


In [5]:
# Fit the selected model using every observed training label.
known = train_raw.loc[train_raw["label"].notna()].copy()
unlabelled = train_raw.loc[train_raw["label"].isna()].copy()

known_ids = known["txId"].tolist()
unlabelled_ids = unlabelled["txId"].tolist()

X_known = get_X_for_ids(train_raw, known_ids)
y_known = known.set_index("txId").loc[known_ids, "label"].astype(int)
X_unlabelled = get_X_for_ids(train_raw, unlabelled_ids)

def create_lgbm_pipeline(params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            objective="binary",
            random_state=100,
            verbosity=-1,
            n_jobs=-1,
            **params,
        )),
    ])

pr_auc_model = create_lgbm_pipeline(pr_auc_params)
log_loss_model = create_lgbm_pipeline(log_loss_params)

pr_auc_model.fit(X_known, y_known)
log_loss_model.fit(X_known, y_known)

p_pr_auc = pr_auc_model.predict_proba(X_unlabelled)[:, 1]
p_log_loss = log_loss_model.predict_proba(X_unlabelled)[:, 1]

p_illicit = (
    weight_log_loss * p_log_loss
    + weight_pr_auc * p_pr_auc
)


In [6]:
# Preserve only the original transaction information and labels,
# excluding all graph-derived node features from the exported file.
output_node_cols = [c for c in node_cols if c in train_raw.columns]
soft_labelled = train_raw.drop(columns=output_node_cols).copy()
soft_labelled["p_illicit"] = np.nan
soft_labelled["label_source"] = "observed"

soft_labelled.loc[unlabelled.index, "p_illicit"] = p_illicit
soft_labelled.loc[unlabelled.index, "label_source"] = "model_probability"

display(soft_labelled.loc[
    unlabelled.index,
    ["txId", "time_step", "label", "p_illicit", "label_source"]
].head())

print(f"Saved {len(unlabelled):,} soft-labelled rows using {model_name}.")
print(f"Weights: {weight_log_loss:.0%} log-loss / {weight_pr_auc:.0%} PR-AUC")
print("Node columns excluded:", output_node_cols)


,txId,time_step,label,p_illicit,label_source
0,230425980,1,NaN,0.002068,model_probability
1,5530458,1,NaN,0.004233,model_probability
2,232022460,1,NaN,0.039134,model_probability
4,230460314,1,NaN,0.000004,model_probability
5,230459870,1,NaN,0.000013,model_probability


Saved 110,537 soft-labelled rows using LightGBM 70/30 ensemble.
Weights: 70% log-loss / 30% PR-AUC
Node columns excluded: ['in_degree', 'out_degree', 'total_degree', 'degree_ratio', 'timestep_tx_volume']


In [7]:
display(
    soft_labelled[["txId", "time_step", "label", "p_illicit", "label_source"]]
    .head(10)
)


,txId,time_step,label,p_illicit,label_source
0,230425980,1,NaN,0.002068,model_probability
1,5530458,1,NaN,0.004233,model_probability
2,232022460,1,NaN,0.039134,model_probability
3,232438397,1,0.0,NaN,observed
4,230460314,1,NaN,0.000004,model_probability
5,230459870,1,NaN,0.000013,model_probability
6,230333930,1,NaN,0.000100,model_probability
7,230595899,1,NaN,0.000936,model_probability
8,232013274,1,NaN,0.001498,model_probability
9,232029206,1,0.0,NaN,observed


In [9]:
soft_labelled.to_csv("train_soft_label.csv", index=False)